In [ ]:
import pandas as pd

class Claim:
    user_id = None
    image_paths = None
    user_claim_transcript = None
    claim_object = None


    #     - `evidence_standard_met`: `true` if the image set is sufficient to evaluate the claim; otherwise `false`
    # - `evidence_standard_met_reason`: short reason for the evidence decision
    # - `risk_flags`: semicolon-separated risk flags, or `none`
    # - `issue_type`: visible issue type
    # - `object_part`: relevant object part
    # - `claim_status`: final decision: `supported`, `contradicted`, or `not_enough_information`
    # - `claim_status_justification`: concise image-grounded explanation; mention relevant image IDs when helpful
    # - `supporting_image_ids`: image IDs supporting the decision, separated by semicolons; use `none` if no image is sufficient
    # - `valid_image`: `true` if the image set is usable for automated review; otherwise `false`
    # - `severity`: `none`, `low`, `medium`, `high`, or `unknown`

    # /------

    # Use the closest matching value from these lists.

    # `claim_status`: `supported`, `contradicted`, `not_enough_information`

    # `issue_type`: `dent`, `scratch`, `crack`, `glass_shatter`, `broken_part`, `missing_part`, `torn_packaging`, `crushed_packaging`, `water_damage`, `stain`, `none`, `unknown`

    # Car `object_part`: `front_bumper`, `rear_bumper`, `door`, `hood`, `windshield`, `side_mirror`, `headlight`, `taillight`, `fender`, `quarter_panel`, `body`, `unknown`

    # Laptop `object_part`: `screen`, `keyboard`, `trackpad`, `hinge`, `lid`, `corner`, `port`, `base`, `body`, `unknown`

    # Package `object_part`: `box`, `package_corner`, `package_side`, `seal`, `label`, `contents`, `item`, `unknown`

    # `risk_flags`: `none`, `blurry_image`, `cropped_or_obstructed`, `low_light_or_glare`, `wrong_angle`, `wrong_object`, `wrong_object_part`, `damage_not_visible`, `claim_mismatch`, `possible_manipulation`, `non_original_image`, `text_instruction_present`, `user_history_risk`, `manual_review_required`

    # Use `issue_type=none` when the relevant part is visible and no issue is present. Use `unknown` when the issue or part cannot be determined.

    #Read the user claim first 

    def evidence_standard_met(self):
        return self.evidence_standard_met
    def evidence_standard_met_reason(self):
        return self.evidence_standard_met_reason
    def risk_flags(self):
        return self.risk_flags
    def issue_type(self):
        return self.issue_type
    def object_part(self):    
        return self.object_part
    def claim_status(self):
        return self.claim_status
    def claim_status_justification(self):
        return self.claim_status_justification
    def supporting_image_ids(self):
        return self.supporting_image_ids
    def valid_image(self):
        return self.valid_image
    def severity(self):
        return self.severity




    

class ProcessClaims(Claim):

    def __init__(self, user_id, image_paths, user_claim_transcript, claim_object):
        self.user_id = user_id
        self.image_paths = image_paths
        self.user_claim_transcript = user_claim_transcript
        self.claim_object = claim_object

    def __repr__(self):
        return f"Claim({self.user_id}, {self.image_paths}, {self.user_claim_transcript}, {self.claim_object})"

In [1]:
from openai import OpenAI

def promptLLM(sysrole, prompt):
    # openai api access 
    client = OpenAI(base_url="http://localhost:1234/v1", api_key="not-needed")

    response = client.chat.completions.create(
        model="gemma-3-1b-it@q8_0",
        messages=[
            {"role": "system", "content": sysrole},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content




In [2]:
import pandas as pd
import json

def MER(image_paths, user_claim_transcript, claim_object):

        valid_rid = []
        valid_rid.append("REQ_GENERAL_OBJECT_PART")
        valid_rid.append("REQ_REVIEW_TRUST")

        if len(image_paths) > 1:
            valid_rid.append("REQ_GENERAL_MULTI_IMAGE")

        if claim_object == "car":
        
            sysrole = "Perform Multi-Label Classification based on the user claim transcript. The output should be a dictionary with the following keys: 'dent or scratch', 'crack, broken, or missing part', 'vehicle identity or orientation'. The values should be 1 if the issue is present and 0 if it is not present. The output should be based on the user claim transcript only. Sample output: { \"dent or scratch\": 1, \"crack, broken, or missing part\": 1, \"vehicle identity or orientation\": 0 }. Important instruction: The output should be in JSON format. Do not include any additional text or explanation outside of the JSON object. "

            output = promptLLM(sysrole, user_claim_transcript)

            # output = json.loads(output.replace('json', '').replace('```', '').replace('\n', ''))
            # Convert the JSON string to a Python dictionary
            #json.dumps()   
            # print(f"Output from LLM: {output.replace('json', '').replace('```', '').replace('\n', '')}")
            # print(f"Output from LLM: {output}")

            output = json.loads(output.replace('json', '').replace('```', '').replace('\n', ''))

            if output['dent or scratch'] == 1:
                valid_rid.append("REQ_CAR_BODY_PANEL")
            if output['crack, broken, or missing part'] == 1:
                valid_rid.append("REQ_CAR_GLASS_LIGHT_MIRROR")
            if output['vehicle identity or orientation'] == 1:
                valid_rid.append("REQ_CAR_IDENTITY_OR_SIDE")

        elif claim_object == "laptop":
            sysrole = "Perform Multi-Label Classification based on the user claim transcript. The output should be a dictionary with the following keys: 'screen, keyboard, or trackpad', 'hinge, lid, corner, body, or port'. The values should be 1 if the issue is present and 0 if it is not present. The output should be based on the user claim transcript only. The output should be in JSON format. Sample output: { \"screen, keyboard, or trackpad\": 1, \"hinge, lid, corner, body, or port\": 0 }. Important instruction: The output should be in JSON format. Do not include any additional text or explanation outside of the JSON object. "

            output = promptLLM(sysrole, user_claim_transcript)
            output = json.loads(output.replace('json', '').replace('```', '').replace('\n', ''))


            if output['screen, keyboard, or trackpad'] == 1:
                valid_rid.append("REQ_LAPTOP_SCREEN_KEYBOARD_TRACKPAD")
            if output['hinge, lid, corner, body, or port'] == 1:
                valid_rid.append("REQ_LAPTOP_BODY_HINGE_PORT")
        
        elif claim_object == "package":
            sysrole = "Perform Multi-Label Classification based on the user claim transcript. The output should be a dictionary with the following keys: 'crushed, torn, or seal damage', 'water, stain, or label damage', 'contents or inner item'. The values should be 1 if the issue is present and 0 if it is not present. The output should be based on the user claim transcript only. The output should be in JSON format. Sample output: { \"crushed, torn, or seal damage\": 1, \"water, stain, or label damage\": 0, \"contents or inner item\": 1 }. Important instruction: The output should be in JSON format. Do not include any additional text or explanation outside of the JSON object. "

            output = promptLLM(sysrole, user_claim_transcript)
            output = json.loads(output.replace('json', '').replace('```', '').replace('\n', ''))


            if output['crushed, torn, or seal damage'] == 1:
                valid_rid.append("REQ_PACKAGE_EXTERIOR")
            if output['water, stain, or label damage'] == 1:
                valid_rid.append("REQ_PACKAGE_LABEL_OR_STAIN")
            if output['contents or inner item'] == 1:
                valid_rid.append("REQ_PACKAGE_CONTENTS")
        
        else:
            valid_rid = []
            raise ValueError("Invalid claim object. Must be 'all', 'car', 'laptop', or 'package'.")
        

        # print(f"Valid requirement IDs: {valid_rid}")

        minimum_image_evidence = ""

        if len(valid_rid) != 0:
            evidence_requirements = pd.read_csv("../dataset/evidence_requirements.csv")
            
            for rid in valid_rid:
                minimum_image_evidence += evidence_requirements.loc[evidence_requirements['requirement_id'] == rid, 'minimum_image_evidence'].values[0] + ";"
            minimum_image_evidence = minimum_image_evidence[:-1]  # Remove the last semicolon
            # print(f"Minimum image evidence required: {minimum_image_evidence}")

        return valid_rid, minimum_image_evidence


In [3]:
sample_claims = pd.read_csv("../dataset/sample_claims.csv")


In [4]:
# for row in range(len(sample_claims)):
#     user = sample_claims.iloc[row]
#     user_id = user['user_id']
#     image_paths = user['image_paths'].split(';')
#     claim_object = user['claim_object']
#     chat_transcript = user['user_claim']
#     try:
#         valid_rid, minimum_image_evidence = test(image_paths, chat_transcript, claim_object)
#         print(f"User ID: {user_id}, Valid Requirement IDs: {valid_rid}, Minimum Image Evidence: {minimum_image_evidence}")
#     except Exception as e:
#         print(f"Error processing user ID {user_id}: {e}")


In [5]:
user = sample_claims[sample_claims['user_id'] == 'user_009'].iloc[0]



In [6]:
valid_rid, minimum_image_evidence = MER(user['image_paths'].split(';'), user['user_claim'], user['claim_object'])
image_paths = user['image_paths'].split(';')

In [7]:
'''
Input: Image, minimum_image_evidence, valid_rid: valid_rid
output: requirement_id : 1 or 0
Sample output: {valid_rid[0]: 1, valid_rid[1]: 0, valid_rid[2]: 1}
'''

def evaluate_requirements(image_paths, minimum_image_evidence, valid_rid):
    evaluation_results = []
    for image_path in image_paths:
        # Structure the prompt here
        # Call the LLM with the structured prompt
        # For demonstration, let's assume the LLM returns a dictionary with requirement_id as keys and 1 or 0 as values
        # In practice, you would replace the following line with the actual call to the LLM and parse its output
        result = {rid: 1 for rid in valid_rid}  # Dummy result for demonstration
        evaluation_results.append(result)
    return evaluation_results



In [8]:
evaluate_requirements(image_paths, minimum_image_evidence, valid_rid)

[{'REQ_GENERAL_OBJECT_PART': 1,
  'REQ_REVIEW_TRUST': 1,
  'REQ_LAPTOP_SCREEN_KEYBOARD_TRACKPAD': 1}]

In [ ]:
import pandas as pd
import json

def evaluate_requirements(image_paths, claim_object):

    valid_rids = []

    for image_path in image_paths:

        valid_rid = []
        valid_rid.append("REQ_GENERAL_OBJECT_PART")
        valid_rid.append("REQ_REVIEW_TRUST")

        if len(image_paths) > 1:
            valid_rid.append("REQ_GENERAL_MULTI_IMAGE")

        if claim_object == "car":
        
            sysrole = "Perform Multi-Label Classification based on the image provided. The output should be a dictionary with the following keys: 'dent or scratch', 'crack, broken, or missing part', 'vehicle identity or orientation'. The values should be 1 if the issue is present and 0 if it is not present. The output should be based on the user claim transcript only. Sample output: { \"dent or scratch\": 1, \"crack, broken, or missing part\": 1, \"vehicle identity or orientation\": 0 }. Important instruction: The output should be in JSON format. Do not include any additional text or explanation outside of the JSON object. "

            output = promptLLM(sysrole, image_path)

            # output = json.loads(output.replace('json', '').replace('```', '').replace('\n', ''))
            # Convert the JSON string to a Python dictionary
            #json.dumps()   
            # print(f"Output from LLM: {output.replace('json', '').replace('```', '').replace('\n', '')}")
            # print(f"Output from LLM: {output}")

            output = json.loads(output.replace('json', '').replace('```', '').replace('\n', ''))

            if output['dent or scratch'] == 1:
                valid_rid.append("REQ_CAR_BODY_PANEL")
            if output['crack, broken, or missing part'] == 1:
                valid_rid.append("REQ_CAR_GLASS_LIGHT_MIRROR")
            if output['vehicle identity or orientation'] == 1:
                valid_rid.append("REQ_CAR_IDENTITY_OR_SIDE")

        elif claim_object == "laptop":
            sysrole = "Perform Multi-Label Classification based on the user claim transcript. The output should be a dictionary with the following keys: 'screen, keyboard, or trackpad', 'hinge, lid, corner, body, or port'. The values should be 1 if the issue is present and 0 if it is not present. The output should be based on the user claim transcript only. The output should be in JSON format. Sample output: { \"screen, keyboard, or trackpad\": 1, \"hinge, lid, corner, body, or port\": 0 }. Important instruction: The output should be in JSON format. Do not include any additional text or explanation outside of the JSON object. "

            output = promptLLM(sysrole, user_claim_transcript)
            output = json.loads(output.replace('json', '').replace('```', '').replace('\n', ''))


            if output['screen, keyboard, or trackpad'] == 1:
                valid_rid.append("REQ_LAPTOP_SCREEN_KEYBOARD_TRACKPAD")
            if output['hinge, lid, corner, body, or port'] == 1:
                valid_rid.append("REQ_LAPTOP_BODY_HINGE_PORT")
        
        elif claim_object == "package":
            sysrole = "Perform Multi-Label Classification based on the user claim transcript. The output should be a dictionary with the following keys: 'crushed, torn, or seal damage', 'water, stain, or label damage', 'contents or inner item'. The values should be 1 if the issue is present and 0 if it is not present. The output should be based on the user claim transcript only. The output should be in JSON format. Sample output: { \"crushed, torn, or seal damage\": 1, \"water, stain, or label damage\": 0, \"contents or inner item\": 1 }. Important instruction: The output should be in JSON format. Do not include any additional text or explanation outside of the JSON object. "

            output = promptLLM(sysrole, user_claim_transcript)
            output = json.loads(output.replace('json', '').replace('```', '').replace('\n', ''))


            if output['crushed, torn, or seal damage'] == 1:
                valid_rid.append("REQ_PACKAGE_EXTERIOR")
            if output['water, stain, or label damage'] == 1:
                valid_rid.append("REQ_PACKAGE_LABEL_OR_STAIN")
            if output['contents or inner item'] == 1:
                valid_rid.append("REQ_PACKAGE_CONTENTS")
        
        else:
            valid_rid = []
            raise ValueError("Invalid claim object. Must be 'all', 'car', 'laptop', or 'package'.")
        

        # print(f"Valid requirement IDs: {valid_rid}")

        minimum_image_evidence = ""

        if len(valid_rid) != 0:
            evidence_requirements = pd.read_csv("../dataset/evidence_requirements.csv")
            
            for rid in valid_rid:
                minimum_image_evidence += evidence_requirements.loc[evidence_requirements['requirement_id'] == rid, 'minimum_image_evidence'].values[0] + ";"
            minimum_image_evidence = minimum_image_evidence[:-1]  # Remove the last semicolon
            # print(f"Minimum image evidence required: {minimum_image_evidence}")

        valid_rids.append(valid_rid)

    return valid_rid, minimum_image_evidence
